# Stage 5 — Ontology Routing Agent

Grounds each **current-visit symptom** to a real SNOMED CT concept, walks its is-a
neighborhood, and extracts the term's natural "concept cluster" using the big-gap
heuristic — replacing Stage 3's LLM-guessed `ontology_hint` with something actually
grounded in the SNOMED CT graph.

**Input** : `patient_records/<patient>/admissions/<hadm>/symptom_tree.json`
**Output**: `patient_records/<patient>/admissions/<hadm>/stage_05_ontology_routing_agent/routed_terms.json`

## Scope for this stage (per project decision)

- Only the **`CURRENT_SYMPTOMS`** branch is routed through SNOMED. `DOCUMENTED_DIAGNOSES`
  (explicit diagnoses named in the current note) and prior-visit history
  (`admission_history.json`, with its own ICD-10 codes) are **carried forward untouched** —
  they're already known, not something to infer, and are reserved for a later
  confidence-scoring stage rather than being re-derived via ontology traversal.
- Symptoms with `status: absent` are skipped (not routed) — same convention as the
  earlier no-LLM Stage 3 draft used.
- **Only cosine similarity is active** as the scoring/cutoff metric right now — it only
  needs local Ollama calls once the neighborhood is found, which keeps a full 15-patient
  batch run practical under the project deadline. **Wu-Palmer is fully implemented
  below but commented out** — the neighborhood (candidate concepts) is still found via
  real is-a graph traversal either way, only the *scoring metric used for the cutoff*
  differs. Re-enabling it later is a matter of uncommenting the marked blocks and
  deciding how to reconcile two independent cutoffs (union, intersection, or picking
  whichever is more confident).
- **LLM-based term normalization runs before grounding** (new). Testing showed several
  real extracted terms (`"abdominal bloatedness"`, `"ICD discharge"`, `"feeling at
  baseline"`) failed to ground because they're colloquial/narrative phrasing rather than
  SNOMED's controlled vocabulary. Before searching SNOMED, each term is passed through
  the same LLM already used elsewhere in this pipeline, asked to reformulate it into
  standard clinical phrasing — or flag it as not a real clinical finding at all.
- Both the interactive Streamlit app (`snomed_similarity_app.py`) and this notebook
  import the same shared module (`notebooks/snomed_ontology.py`), so search, graph
  traversal, Wu-Palmer, and the big-gap heuristic only exist in one place.


## 1. Setup

In [1]:
import sys
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import requests

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from snomed_ontology import (
    configure,
    configure_bioportal,
    load_umls_api_key,
    load_bioportal_api_key,
    search_snomed,
    search_snomed_bioportal,
    get_sctid,
    get_cui_for_sctid,
    get_concept_name,
    get_semantic_tag,
    GROUNDABLE_SEMANTIC_TAGS,
    explore_neighborhood,
    wu_palmer,  # imported for the commented-out Wu-Palmer block below
    biggest_gap_cutoff,
    embed,
    cosine_sim,
    MAX_WORKERS,
)
from pipeline import (
    LLMNotAvailableError,
    call_llm_json,
    check_llm,
    get_llm_config,
    print_pipeline_banner,
)

PROJECT_ROOT = NB_DIR.parent
RECORDS_DIR  = PROJECT_ROOT / "patient_records"
STAGE_OUTPUT = "stage_05_ontology_routing_agent"
CACHE_FILE   = RECORDS_DIR / "ontology_routing_cache.json"
MAX_HOPS     = 3

UMLS_API_KEY = load_umls_api_key(PROJECT_ROOT)
configure(UMLS_API_KEY)
print(f"UMLS key loaded: {bool(UMLS_API_KEY)}")

BIOPORTAL_API_KEY = load_bioportal_api_key(PROJECT_ROOT)
configure_bioportal(BIOPORTAL_API_KEY)
print(f"BioPortal key loaded: {bool(BIOPORTAL_API_KEY)}")

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready for term normalization — {LLM_CONFIG.method_prefix()}: {model_info}")

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Patients found: {len(patients)}")


UMLS key loaded: True
BioPortal key loaded: True
Pipeline mode : FULL (15 patients)
LLM provider  : Ollama (qwen2.5:7b)
Qwen pair     : Cloud equivalent: qwen/qwen-2.5-7b-instruct on OpenRouter
Admissions/patient (min): 2
Data dir      : C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\data
Export dir    : C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records
LLM ready for term normalization — ollama: qwen2.5:7b
Patients found: 15


## 2. Term-level cache

Mirrors the `snomed_cache.json` pattern already used in `stage_06_snomed_grounding.ipynb`.
The same symptom term ("chest pain", "acute kidney injury", ...) shows up across many
admissions — caching by the literal raw term string avoids re-running normalization and
re-hitting UMLS/Ollama for repeats, and makes the batch run resumable if it's interrupted
partway through.

In [2]:
if CACHE_FILE.exists():
    with open(CACHE_FILE, encoding="utf-8") as f:
        TERM_CACHE = json.load(f)
    print(f"Cache loaded: {len(TERM_CACHE)} terms already routed")
else:
    TERM_CACHE = {}
    print("No cache yet — will build one as terms are routed")


def save_cache():
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(TERM_CACHE, f, indent=2)


Cache loaded: 74 terms already routed


## 3. Term normalization (LLM)

Reformulates a raw extracted term into standard clinical phrasing *before* attempting to
ground it in SNOMED CT — bridging the gap between colloquial/narrative extracted language
and SNOMED's controlled vocabulary. Also catches phrases that aren't clinical findings at
all (e.g. "feeling at baseline") so they're skipped rather than force-mapped to something
wrong.

Deliberately conservative: it's asked to normalize *wording*, not add diagnostic
interpretation beyond what's stated — e.g. "shaking in his chest" should become something
closer to its literal description, not a specific diagnosis the note itself doesn't assert.

In [3]:
NORMALIZE_TERM_SYSTEM_PROMPT = """You are a clinical terminology normalizer. Given a symptom
or finding phrase extracted from a clinical note, along with the verbatim evidence quote it
came from, return the standard clinical term that best names this finding \u2014 the kind of
phrasing that would appear as a SNOMED CT preferred term or synonym.

Rules:
- Prefer standard clinical terminology over colloquial phrasing (e.g. "abdominal
  bloatedness" -> "Abdominal bloating").
- Normalize wording only \u2014 do not add a more specific diagnosis or interpretation than
  what the evidence quote actually supports.
- If the phrase does not describe an actual clinical finding (e.g. "feeling at baseline",
  "no complaints", administrative or narrative text with no clinical content), set
  "groundable" to false and leave "normalized_term" empty.

Return ONLY valid JSON:
{
  "normalized_term": "standardized clinical term, or empty string if not groundable",
  "groundable": true or false,
  "reasoning": "one short phrase explaining the change, or why it's not groundable"
}"""


def normalize_term(term: str, evidence: str, config) -> dict:
    """LLM-based normalization step, run once per unique raw term (cached alongside
    the rest of that term's routing result)."""
    user_prompt = f'Extracted term: "{term}"\nEvidence quote: "{evidence}"'
    try:
        result = call_llm_json(NORMALIZE_TERM_SYSTEM_PROMPT, user_prompt, config)
    except (LLMNotAvailableError, ValueError) as e:
        # Normalization is a nice-to-have, not a hard dependency -- if the LLM call
        # itself fails, fall back to grounding the raw term rather than losing it.
        return {"normalized_term": term, "groundable": True, "reasoning": f"normalization failed, using raw term: {e}"}

    normalized = str(result.get("normalized_term", "")).strip()
    groundable = bool(result.get("groundable", True))
    if not normalized:
        groundable = False
    return {
        "normalized_term": normalized if groundable else "",
        "groundable": groundable,
        "reasoning": result.get("reasoning", ""),
    }


# Sanity check against the real failures found during testing.
for _term, _evidence in [
    ("abdominal bloatedness", "abdominal bloatedness secondary to gas"),
    ("ICD discharge", "ICD discharge"),
    ("feeling at baseline", "felt at baseline"),
]:
    print(_term, "->", normalize_term(_term, _evidence, LLM_CONFIG))


abdominal bloatedness -> {'normalized_term': 'Abdominal bloating', 'groundable': True, 'reasoning': "Replaced 'bloatedness' with the standard term 'bloating'."}
ICD discharge -> {'normalized_term': '', 'groundable': False, 'reasoning': 'This phrase does not describe a clinical finding.'}
feeling at baseline -> {'normalized_term': '', 'groundable': False, 'reasoning': 'This phrase does not describe a clinical finding.'}


## 4. Grounding

Resolves a (normalized) free-text term to a real SNOMED CT concept. Unlike the interactive
Streamlit app — where you pick the right match from a radio list — this is a batch pipeline
with no human in the loop, so disambiguation has to be automatic: `search_snomed()` always
orders an exact-match hit first when one exists (see its exact+fuzzy backstop), so taking
the top result is a reasonable default.

In [5]:
def ground_term(term: str):
    """Resolve `term` to a SNOMED CT concept.

    Combines candidates from both BioPortal and UMLS search -- each has
    retrieval gaps the other doesn't. Tested empirically on "Vaginal
    hemorrhage": BioPortal alone never retrieved the correct concept in its
    top 8 at all, while UMLS did (at rank 2). Combining both maximizes the
    chance the right concept is somewhere in the pool.

    Candidates are deduplicated by *resolved SCTID*, not by name string. This
    matters because SNOMED CT routinely has two genuinely different concepts
    -- a clinical disorder and its own morphology/histology concept -- sharing
    the identical preferred term. Real case found via UMLS's own CUI clustering:
    CUI C2239176 groups both "Liver cell carcinoma" (109841003, disorder) and
    "Hepatocellular carcinoma" (25370001, morphologic abnormality) as
    synonymous atoms, because the text is genuinely interchangeable in clinical
    writing even though SNOMED itself models them as separate concepts.
    Deduplicating by name string before resolving SCTIDs let whichever source
    returned "Hepatocellular carcinoma" *first* (BioPortal, which indexes the
    morphology concept directly under that exact text) silently claim the name
    slot and block the disorder-tagged sibling from ever entering the pool as
    its own candidate -- so the semantic-tag filter below never got a chance to
    choose between them, because only one of the two ever arrived. Resolving
    every candidate's SCTID first and deduplicating on that identity instead
    lets both distinct concepts survive into the pool, so the filter can do
    its actual job.

    Before ranking, the (SCTID-deduplicated) pool is filtered to only those
    SNOMED tags as (disorder) or (finding) -- SNOMED's own categorization of
    "something a patient can actually have/present with". If filtering would
    eliminate every candidate, falls back to the unfiltered pool rather than
    losing the grounding entirely.

    The remaining pool is then re-ranked by cosine similarity to the query
    term, rather than trusting either API's own relevance ranking -- neither
    is clinically aware. On the "Vaginal hemorrhage" test term, both
    independently ranked the narrower, age-inappropriate "Neonatal vaginal
    hemorrhage" above the correct general concept; cosine similarity
    correctly separates them (1.0000 vs 0.8460 in testing).

    Returns None if nothing was found in either source.
    """
    umls_candidates = search_snomed(term)
    bioportal_candidates = search_snomed_bioportal(term)

    raw_pool = list(bioportal_candidates) + list(umls_candidates)
    if not raw_pool:
        return None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Resolve every candidate's SCTID up front -- BioPortal candidates
        # already have one, UMLS candidates only have a CUI at this point.
        # This has to happen *before* dedup now, since dedup keys off the
        # resolved SCTID rather than the candidate's display name.
        sctids = list(executor.map(lambda c: c.get("sctid") or get_sctid(c.get("cui", "")), raw_pool))

        pool = []
        seen_sctids = set()
        for c, sctid in zip(raw_pool, sctids):
            if sctid and sctid not in seen_sctids:
                seen_sctids.add(sctid)
                pool.append({**c, "sctid": sctid})

        if not pool:
            return None

        # -- ACTIVE: semantic-tag filtering (disorder/finding only) ------------
        tags = list(executor.map(lambda c: get_semantic_tag(c["sctid"]), pool))

    groundable_pool = [c for c, tag in zip(pool, tags) if tag in GROUNDABLE_SEMANTIC_TAGS]
    if groundable_pool:
        pool = groundable_pool
    # else: nothing passed the filter -- fall back to the full unfiltered pool
    # rather than returning no grounding at all.

    if len(pool) == 1:
        picked = pool[0]
    else:
        try:
            qvec = embed(term)
            picked = max(pool, key=lambda c: cosine_sim(qvec, embed(c["name"])))
        except requests.RequestException:
            # Ollama unavailable -- fall back to whichever candidate came first
            picked = pool[0]

    sctid = picked.get("sctid")
    if not sctid:
        return None
    cui = picked.get("cui") or get_cui_for_sctid(sctid)
    return {"cui": cui, "sctid": sctid, "name": picked["name"]}


# Quick sanity check against the exact term that motivated the cosine-rerank fix.
print(ground_term("Vaginal hemorrhage"))

# Sanity check against the semantic-tag fix: should now avoid the bare
# morphology concept (25370001) and land on the actual disorder concept.
print(ground_term("Multifocal metastatic HCC"))

# Sanity check against the name-dedup fix: this is the term the real pipeline
# actually grounds after LLM normalization ("Multifocal metastatic HCC" ->
# "Hepatocellular carcinoma"). Previously landed on 25370001 (morphologic
# abnormality) because BioPortal's identically-named morphology candidate blocked
# the disorder-tagged sibling from ever entering the pool. Should now land on a
# disorder-tagged SCTID (109841003 "Liver cell carcinoma" or a similarly-tagged
# sibling), not 25370001.
print(ground_term("Hepatocellular carcinoma"))


{'cui': 'C2979982', 'sctid': '289530006', 'name': 'Vaginal Hemorrhage'}
{'cui': 'C5816701', 'sctid': '1363113005', 'name': 'Multifocal tuberculosis'}


## 5. Routing -- normalize, ground, explore the neighborhood, score, apply the big-gap cutoff

**Active metric: cosine similarity** (`nomic-embed-text` embeddings), used for every
scoring step in this notebook -- grounding (in `ground_term`) and neighborhood scoring
(below) both compare concepts via the same metric now. Wu-Palmer (graph-structural,
is-a hierarchy) is fully implemented below but commented out, kept for a future
extension.

Grounding is now attempted on the **normalized** term first; if that fails to find a
SNOMED match, it falls back to trying the **raw** term too, before giving up -- cheap
insurance against normalization occasionally making things worse.


In [6]:
def route_term(term: str, evidence: str = "", max_hops: int = MAX_HOPS):
    """Normalize `term`, ground it in SNOMED, walk its is-a neighborhood, score every
    discovered concept with cosine similarity, and apply the big-gap cutoff to
    extract its natural cluster. Cached per literal raw term string.

    Grounding (picking *which* concept a term maps to) and neighborhood scoring
    (comparing the already-grounded seed to its graph neighbors) both use cosine
    similarity now, so there's only one metric and one cutoff mechanism in play
    throughout this notebook. Wu-Palmer (graph-structural, is-a hierarchy) is fully
    implemented below but commented out -- kept for a future extension, e.g. if a
    later evaluation stage shows graph distance catches cases cosine misses.
    """
    if term in TERM_CACHE:
        return TERM_CACHE[term]

    norm = normalize_term(term, evidence, LLM_CONFIG)

    if not norm["groundable"]:
        result = {
            "raw_term": term,
            "normalized_term": norm["normalized_term"],
            "normalization_reasoning": norm["reasoning"],
            "grounded": None,
            "grounded_via": None,
            "neighborhood": None,
        }
        TERM_CACHE[term] = result
        return result

    # Try the normalized term first, fall back to the raw term if that doesn't ground.
    grounded = ground_term(norm["normalized_term"])
    grounded_via = "normalized"
    if grounded is None or not grounded["sctid"]:
        grounded = ground_term(term)
        grounded_via = "raw" if grounded else None

    if grounded is None or not grounded["sctid"]:
        result = {
            "raw_term": term,
            "normalized_term": norm["normalized_term"],
            "normalization_reasoning": norm["reasoning"],
            "grounded": None,
            "grounded_via": None,
            "neighborhood": None,
        }
        TERM_CACHE[term] = result
        return result

    seed_sctid = grounded["sctid"]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        found = explore_neighborhood(seed_sctid, max_hops, executor)
        candidate_sctids = list(found)

        if not candidate_sctids:
            result = {
                "raw_term": term,
                "normalized_term": norm["normalized_term"],
                "normalization_reasoning": norm["reasoning"],
                "grounded": grounded,
                "grounded_via": grounded_via,
                "neighborhood": [],
            }
            TERM_CACHE[term] = result
            return result

        names = list(executor.map(get_concept_name, candidate_sctids))

        # -- ACTIVE: cosine similarity (text-embedding, nomic-embed-text) -----------
        seed_vec = embed(grounded["name"])
        candidate_vecs = list(executor.map(embed, names))
        cos_scores = [round(cosine_sim(seed_vec, v), 4) for v in candidate_vecs]

        # -- COMMENTED OUT -- Wu-Palmer similarity (graph-structural, is-a hierarchy) --
        # Uncomment to compute the graph-distance metric alongside/instead of cosine.
        # Needs no embeddings/Ollama at all, just repeated /parents /children calls
        # (already cached via ancestors_with_dist).
        # wp_scores = [wu_palmer(seed_sctid, sctid) or 0.0 for sctid in candidate_sctids]

    rows = list(zip(candidate_sctids, names, cos_scores))
    rows.sort(key=lambda r: r[2], reverse=True)  # sort by cosine -- biggest_gap_cutoff needs it sorted

    scored = [(sctid, cos) for sctid, _, cos in rows]
    cutoff_idx, gap, is_significant = biggest_gap_cutoff(scored)
    n_kept = (cutoff_idx + 1) if cutoff_idx is not None else len(rows)

    # -- COMMENTED OUT -- Wu-Palmer's own independent big-gap cutoff ---------------
    # wp_rows = list(zip(candidate_sctids, names, wp_scores))
    # wp_rows.sort(key=lambda r: r[2], reverse=True)
    # wp_scored = [(sctid, wp) for sctid, _, wp in wp_rows]
    # wp_cutoff_idx, wp_gap, wp_is_significant = biggest_gap_cutoff(wp_scored)
    # wp_n_kept = (wp_cutoff_idx + 1) if wp_cutoff_idx is not None else len(wp_rows)

    result = {
        "raw_term": term,
        "normalized_term": norm["normalized_term"],
        "normalization_reasoning": norm["reasoning"],
        "grounded": grounded,
        "grounded_via": grounded_via,
        "hops_explored": max_hops,
        "n_candidates": len(rows),
        "cosine_cutoff": {"gap": gap, "is_significant": is_significant, "n_kept": n_kept},
        "neighborhood": [
            {
                "sctid": sctid,
                "name": name,
                "cosine_similarity": cos,
                "hops": found[sctid],
                "in_cluster": i < n_kept,
            }
            for i, (sctid, name, cos) in enumerate(rows)
        ],
        # "wu_palmer_cutoff": {  # -- COMMENTED OUT -- see above --
        #     "gap": wp_gap, "is_significant": wp_is_significant, "n_kept": wp_n_kept,
        # },
    }
    TERM_CACHE[term] = result
    return result


## 6. Test on real terms before running the full batch

Includes the three terms that failed to ground before normalization was added — this is
the regression check that the fix actually works, not just that the pipeline runs.

In [7]:
for _term, _evidence in [
    ("chest pain", "chest pain"),
    ("abdominal bloatedness", "abdominal bloatedness secondary to gas"),
    ("ICD discharge", "ICD discharge"),
]:
    routing_result = route_term(_term, _evidence)
    print(f"Raw term: {routing_result['raw_term']}")
    print(f"Normalized: {routing_result['normalized_term']!r}  ({routing_result['normalization_reasoning']})")
    print(f"Grounded: {routing_result['grounded']}  (via: {routing_result['grounded_via']})")
    if routing_result.get("cosine_cutoff"):
        print(f"Cosine cutoff: {routing_result['cosine_cutoff']}")
    print()


Raw term: chest pain
Normalized: 'Chest pain'  (Exact match with standard clinical terminology)
Grounded: {'cui': 'C0008031', 'sctid': '29857009', 'name': 'Chest pain'}  (via: normalized)

Raw term: abdominal bloatedness
Normalized: 'Abdominal bloating'  (Replaced 'bloatedness' with the standard term 'bloating'.)
Grounded: {'cui': 'C1291077', 'sctid': '116289008', 'name': 'Abdominal bloating'}  (via: normalized)

Raw term: ICD discharge
Normalized: ''  (This phrase does not describe a clinical finding.)
Grounded: None  (via: None)



## 7. Run across all patients

Only `CURRENT_SYMPTOMS` (non-absent) terms are routed. `DOCUMENTED_DIAGNOSES` and prior
admission history are carried forward as-is — reserved for a later scoring stage, not
routed through SNOMED. Cache is saved after every admission so an interruption doesn't
lose progress.

In [8]:
processed_admissions = 0
processed_terms = 0
failed_terms = []  # (patient_id, admission_id, term, error) -- routing failures that didn't crash the run

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    adm_root = patient_dir / "admissions"
    adm_dirs = sorted(adm_root.iterdir()) if adm_root.exists() else []

    for adm_dir in adm_dirs:
        tree_path = adm_dir / "symptom_tree.json"
        if not tree_path.exists():
            print(f"  SKIP {patient_id}/{adm_dir.name} -- no symptom_tree.json")
            continue

        with open(tree_path, encoding="utf-8") as f:
            tree = json.load(f)

        routed_branches = []
        carried_forward_branches = []

        for branch in tree.get("branches", []):
            category = branch.get("category", "")

            if category != "CURRENT_SYMPTOMS":
                # DOCUMENTED_DIAGNOSES (and anything else) is already a known
                # diagnosis -- carried forward untouched, reserved for a later
                # confidence-scoring stage rather than routed through SNOMED.
                carried_forward_branches.append(branch)
                continue

            routed_symptoms = []
            for symptom in branch.get("symptoms", []):
                if str(symptom.get("status", "unknown")).lower() == "absent":
                    routed_symptoms.append({**symptom, "routing": None})
                    continue
                term = symptom.get("term", "").strip()
                if not term:
                    continue
                evidence = symptom.get("evidence", "")

                # A single bad term (LLM hiccup, UMLS timeout, unexpected API
                # response, etc.) shouldn't take down a 15-patient batch run --
                # record the failure and move on rather than crashing partway through.
                try:
                    routing = route_term(term, evidence)
                except Exception as e:
                    print(f"  ERROR routing {term!r} ({patient_id}/{adm_dir.name}): {e}")
                    failed_terms.append((patient_id, adm_dir.name, term, str(e)))
                    routing = {"raw_term": term, "grounded": None, "neighborhood": None, "error": str(e)}

                routed_symptoms.append({**symptom, "routing": routing})
                processed_terms += 1

            routed_branches.append({**branch, "symptoms": routed_symptoms})

        output = {
            "patient_id": tree.get("patient_id"),
            "admission_id": tree.get("admission_id"),
            "routed_branches": routed_branches,                    # CURRENT_SYMPTOMS, SNOMED-grounded
            "carried_forward_branches": carried_forward_branches,  # DOCUMENTED_DIAGNOSES, untouched
            "key_symptoms": tree.get("key_symptoms", []),
            "red_flags": tree.get("red_flags", []),
        }

        out_dir = adm_dir / STAGE_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "routed_terms.json", "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2)

        save_cache()  # persist after every admission -- a crash shouldn't lose progress
        processed_admissions += 1
        n_sym = len(routed_branches[0]["symptoms"]) if routed_branches else 0
        print(f"Patient {patient_id} | {adm_dir.name} | {n_sym} symptoms routed")

print(f"\nDone. {processed_admissions} admissions processed, {processed_terms} symptom routings, "
      f"cache now has {len(TERM_CACHE)} entries.")
if failed_terms:
    print(f"\n{len(failed_terms)} term(s) failed and were recorded with an error instead of a routing -- review these:")
    for patient_id, adm_name, term, err in failed_terms:
        print(f"  {patient_id}/{adm_name}: {term!r} -> {err}")


Patient 10361982 | hadm_24286431 | 3 symptoms routed
Patient 10426859 | hadm_29908281 | 5 symptoms routed
Patient 10458324 | hadm_21744342 | 3 symptoms routed
Patient 11251337 | hadm_29568708 | 4 symptoms routed
Patient 11474876 | hadm_29672491 | 4 symptoms routed
Patient 11607177 | hadm_23293838 | 5 symptoms routed
Patient 12007928 | hadm_23749816 | 7 symptoms routed
Patient 13196707 | hadm_21475988 | 6 symptoms routed
Patient 13508515 | hadm_21834271 | 4 symptoms routed
Patient 13952483 | hadm_23852410 | 13 symptoms routed
Patient 16014068 | hadm_29042843 | 4 symptoms routed
Patient 17774110 | hadm_27339772 | 11 symptoms routed
Patient 18412100 | hadm_26093939 | 6 symptoms routed
Patient 19104262 | hadm_24271247 | 2 symptoms routed
Patient 19632936 | hadm_26696232 | 10 symptoms routed

Done. 15 admissions processed, 78 symptom routings, cache now has 74 entries.


## 8. Inspect one admission

In [15]:
EXAMPLE_IDX = 10
example_patient = patients[EXAMPLE_IDX]
adm_dirs = sorted((example_patient / "admissions").iterdir())
routed_path = adm_dirs[0] / STAGE_OUTPUT / "routed_terms.json"

with open(routed_path, encoding="utf-8") as f:
    example = json.load(f)

print(f'Patient {example["patient_id"]} | Admission {example["admission_id"]}')
print()
for branch in example["routed_branches"]:
    print(f'BRANCH: {branch["category"]}')
    for s in branch["symptoms"]:
        routing = s.get("routing")
        if routing is None:
            print(f'  [absent]  {s["term"]}')
            continue
        grounded = routing.get("grounded")
        normalized = routing.get("normalized_term", "")
        n_kept = routing.get("cosine_cutoff", {}).get("n_kept", "?")
        norm_note = f' (normalized: "{normalized}")' if normalized and normalized != s["term"] else ""
        print(f'  {s["term"]:35s}{norm_note} -> '
              f'{grounded["name"] if grounded else "NOT GROUNDED":35s} (cluster size: {n_kept})')

print()
print(f'Carried forward untouched: {len(example["carried_forward_branches"])} branch(es)')
for branch in example["carried_forward_branches"]:
    print(f'  {branch["category"]}: {[s["term"] for s in branch["symptoms"]]}')


Patient 16014068 | Admission 29042843

BRANCH: CURRENT_SYMPTOMS
  weakness                            (normalized: "Weakness") -> Weakness                            (cluster size: ?)
  minor abrasions and soreness        (normalized: "Minor abrasions and skin irritation") -> Skin irritation                     (cluster size: ?)
  headache                            (normalized: "Headache") -> Headache                            (cluster size: ?)
  intracranial bleeding               (normalized: "Intracranial hemorrhage") -> Intracranial hemorrhage             (cluster size: ?)

Carried forward untouched: 1 branch(es)
  DOCUMENTED_DIAGNOSES: ['metastatic sarcoma to the brain', 'right femur x-rays showed post-surgical changes without complications', 'pulmonary nodules', 'pulmonary emboli']
